# Spark ETL Operator Wrapper Test

Minimal notebook for Enterprise Gateway `spark_python_operator` kernel.
Run cells top-to-bottom.


In [ ]:
import os
import socket
import sys
from pathlib import Path

cwd = Path.cwd()
print(f"Host: {socket.gethostname()}")
print(f"CWD: {cwd}")
print(f"KERNEL_ID: {os.getenv('KERNEL_ID')}")
print(f"KERNEL_USERNAME: {os.getenv('KERNEL_USERNAME')}")

# EG Spark kernels may not inherit Spark's Python paths in sys.path.
spark_py = Path('/opt/spark/python')
spark_lib = spark_py / 'lib'
for p in [spark_py, spark_lib / 'pyspark.zip', *sorted(spark_lib.glob('py4j-*-src.zip'))]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from pyspark.sql import SparkSession  # noqa: F401
    print('PySpark import: ok')
except Exception as exc:
    raise RuntimeError('PySpark is not available in this kernel. Use EG kernel Spark Operator (Python).') from exc

src_dir = Path('/opt/spark/jobs')
if not (src_dir / 'notebook_wrapper.py').exists():
    raise RuntimeError(
        "notebook_wrapper.py not found at /opt/spark/jobs. Ensure the SparkApplication template mounts"
        " ConfigMap spark-etl-operator-scripts to /opt/spark/jobs."
    )

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Using wrapper source: {src_dir}")


In [ ]:
import os

# Optional: override defaults for your environment
os.environ.setdefault('MYSQL_HOST', 'mysql-service.data-simulator.svc.cluster.local')
os.environ.setdefault('MYSQL_PORT', '3306')
os.environ.setdefault('MYSQL_DATABASE', 'telecom_data')
os.environ.setdefault('MYSQL_USER', 'telecom_user')
os.environ.setdefault('MYSQL_PASSWORD', 'telecom_password')

os.environ.setdefault('S3_BUCKET', 'telecom-cdr-data')
os.environ.setdefault('S3_ENDPOINT', 'http://minio-service.minio.svc.cluster.local:9000')
os.environ.setdefault('S3_ACCESS_KEY', 'minio')
os.environ.setdefault('S3_SECRET_KEY', 'minio123')

os.environ.setdefault('EXTRACT_MODE', 'incremental')
os.environ.setdefault('INCREMENTAL_HOURS', '1')

# This demo supports only the baked kernel image runtime.
# Do not inject custom Spark jar overrides from notebook env.
for env_name in ('SPARK_JARS', 'SPARK_JARS_PACKAGES', 'SPARK_AUTO_INCLUDE_MYSQL_JDBC'):
    os.environ.pop(env_name, None)


In [ ]:
from notebook_wrapper import create_spark, run_analytics, run_etl

spark = create_spark('cdr-wrapper-notebook')
spark


In [ ]:
RUN_ETL = False
RUN_ANALYTICS = False

etl_result = None
analytics_result = None

if RUN_ETL:
    etl_result = run_etl(spark)
    etl_result

if RUN_ANALYTICS:
    analytics_result = run_analytics(spark)
    analytics_result


In [ ]:
if 'spark' in globals() and spark is not None:
    spark.stop()
'Spark session stopped'
